# Sistema de Recomendação de Produtos Financeiros
## Abordagem: Logistic Regression + Causal Inference (Position Debiasing) + Epsilon-Greedy

**Banco Digital Brasileiro — Case Técnico de Ciência de Dados**

---

## 0. Setup e Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings, json, pickle
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Setup concluido.')

## 1. Carregamento dos Dados

In [ ]:
DATA_PATH = 'data'  # ajuste para o path local

clientes   = pd.read_csv(f'{DATA_PATH}/clientes.csv')
interacoes = pd.read_csv(f'{DATA_PATH}/interacoes.csv', parse_dates=['timestamp'])
contratos  = pd.read_csv(f'{DATA_PATH}/contratos_ativos.csv', parse_dates=['data_contratacao'])
produtos   = pd.read_csv(f'{DATA_PATH}/produtos.csv')

print(f'Clientes:   {clientes.shape}')
print(f'Interacoes: {interacoes.shape}')
print(f'Contratos:  {contratos.shape}')
print(f'Produtos:   {produtos.shape}')
clientes.head(3)

## 2. Analise Exploratoria dos Dados (EDA)

### 2.1 Distribuicao de Clientes por Segmento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

seg = clientes['segmento'].value_counts()
axes[0].bar(seg.index, seg.values, color=['#4C72B0','#DD8452','#55A868'])
axes[0].set_title('Clientes por Segmento')
axes[0].set_ylabel('Quantidade')

axes[1].hist(clientes['idade'], bins=20, color='#4C72B0', edgecolor='white')
axes[1].set_title('Distribuicao de Idade')
axes[1].set_xlabel('Idade')

axes[2].hist(clientes['renda_mensal'].clip(upper=20000), bins=30, color='#DD8452', edgecolor='white')
axes[2].set_title('Distribuicao de Renda (ate R$20k)')
axes[2].set_xlabel('Renda Mensal (R$)')

plt.tight_layout()
plt.show()

print(clientes[['segmento','score_credito','renda_mensal','saldo_medio_conta']].groupby('segmento').mean().round(2))

### 2.2 Vies de Posicao no Carrossel

**Este e o achado mais importante do EDA e a motivacao principal para Causal Inference.**

A taxa de conversao cai drasticamente com a posicao de exibicao. Sem controlar esse efeito,
o modelo aprenderia que 'estar na posicao 1 causa contratacao' — um loop vicioso que perpetuaria
a politica de exibicao estatica atual em vez de aprender preferencias reais dos clientes.

In [ ]:
pos_agg = interacoes.groupby('posicao_exibicao').agg(
    n=('id_interacao','count'),
    contratos=('contratou','sum')
).reset_index()
pos_agg['conversion_rate'] = pos_agg['contratos'] / pos_agg['n']

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(pos_agg['posicao_exibicao'], pos_agg['conversion_rate'] * 100, color='#4C72B0', alpha=0.8)
ax.axvline(5.5, color='red', linestyle='--', linewidth=2, label='Limite visivel sem scroll')
ax.set_xlabel('Posicao no Carrossel')
ax.set_ylabel('Taxa de Conversao (%)')
ax.set_title('Taxa de Conversao por Posicao — Vies de Posicao Evidente')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Conversao posicao 1:  {pos_agg.loc[pos_agg.posicao_exibicao==1, 'conversion_rate'].values[0]:.3%}")
print(f"Conversao posicao 10: {pos_agg.loc[pos_agg.posicao_exibicao==10, 'conversion_rate'].values[0]:.3%}")
print(f"Conversao posicao 20: {pos_agg.loc[pos_agg.posicao_exibicao==20, 'conversion_rate'].values[0]:.3%}")

### 2.3 Taxa de Conversao por Produto e Canal

In [ ]:
agg_produto = interacoes.groupby('produto').agg(
    impressoes=('id_interacao','count'),
    cliques=('clicou','sum'),
    contratos=('contratou','sum'),
    receita=('receita_gerada','sum'),
).reset_index()
agg_produto['ctr'] = agg_produto['cliques'] / agg_produto['impressoes']
agg_produto['conversion_rate'] = agg_produto['contratos'] / agg_produto['impressoes']
agg_produto = agg_produto.sort_values('contratos', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(agg_produto)))
ax1.barh(agg_produto['produto'], agg_produto['contratos'], color=colors)
ax1.set_title('Total de Contratacoes por Produto')
ax1.set_xlabel('Contratacoes')
ax2.barh(agg_produto.sort_values('conversion_rate')['produto'],
         agg_produto.sort_values('conversion_rate')['conversion_rate'] * 100, color=colors)
ax2.set_title('Taxa de Conversao por Produto (%)')
ax2.set_xlabel('Taxa de Conversao (%)')
plt.tight_layout()
plt.show()

canal_agg = interacoes.groupby('canal').agg(
    impressoes=('id_interacao','count'),
    contratos=('contratou','sum'),
).reset_index()
canal_agg['conversion_rate'] = canal_agg['contratos'] / canal_agg['impressoes']
print('Taxa de conversao por canal:')
print(canal_agg.sort_values('conversion_rate', ascending=False)[['canal','conversion_rate']].to_string(index=False))

### 2.4 Analise Temporal — Contratacoes por Safra

In [ ]:
safra_agg = interacoes.groupby('safra').agg(
    contratos=('contratou','sum'),
    impressoes=('id_interacao','count'),
).reset_index()
safra_agg['conversion_rate'] = safra_agg['contratos'] / safra_agg['impressoes']

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(safra_agg['safra'].astype(str), safra_agg['contratos'], marker='o', linewidth=2, color='#4C72B0')
idx_cutoff = safra_agg[safra_agg['safra']==202509].index[0]
ax.axvline(x=idx_cutoff, color='red', linestyle='--', label='Cutoff treino/teste (202509)')
ax.set_title('Contratacoes por Safra')
ax.set_xlabel('Safra')
ax.set_ylabel('Contratacoes')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.5 Analise de Co-contratacao

In [ ]:
ativos_cliente = contratos[contratos.status=='ativo'].groupby('id_cliente')['produto'].apply(list)
co_pairs = []
for prods in ativos_cliente:
    prods = sorted(set(prods))
    for i in range(len(prods)):
        for j in range(i+1, len(prods)):
            co_pairs.append((prods[i], prods[j]))

co_df = pd.DataFrame(co_pairs, columns=['p1','p2'])
top_co = co_df.groupby(['p1','p2']).size().sort_values(ascending=False).head(10)
print('Top 10 pares de co-contratacao:')
print(top_co.to_string())

## 3. Tratamento de Dados e Feature Engineering

In [ ]:
# Imputacao de missings
print('Missings antes:', clientes.isnull().sum().sum())
clientes['vlr_total_investimentos'] = clientes['vlr_total_investimentos'].fillna(0)
clientes['qtd_dias_inatividade']    = clientes['qtd_dias_inatividade'].fillna(clientes['qtd_dias_inatividade'].median())
print('Missings depois:', clientes.isnull().sum().sum())

# Encoding
for col in ['segmento', 'canal_preferencial', 'genero']:
    le = LabelEncoder()
    clientes[f'{col}_enc'] = le.fit_transform(clientes[col])

# Features derivadas (razoes financeiras)
clientes['renda_saldo_ratio']     = clientes['saldo_medio_conta'] / (clientes['renda_mensal'] + 1)
clientes['gasto_renda_ratio']     = clientes['vlr_medio_gasto_cartao'] / (clientes['renda_mensal'] + 1)
clientes['inv_renda_ratio']       = clientes['vlr_total_investimentos'] / (clientes['renda_mensal'] + 1)
clientes['utilizacao_limite_bin'] = (clientes['pct_utilizacao_limite'] > 0.7).astype(int)
clientes['cliente_antigo']        = (clientes['qtd_meses_cliente'] > 24).astype(int)
clientes['alta_renda']            = (clientes['renda_mensal'] > clientes['renda_mensal'].quantile(0.8)).astype(int)

print('Feature engineering de clientes concluida. Shape:', clientes.shape)

In [ ]:
# Estatisticas historicas por produto
prod_stats = interacoes.groupby('produto').agg(
    prod_conv_rate=('contratou','mean'),
    prod_clique_rate=('clicou','mean'),
).reset_index()
produtos = produtos.merge(prod_stats, on='produto', how='left')
cat_map = produtos.set_index('produto')['categoria'].to_dict()

# Historico de interacoes por (cliente, produto)
cp_hist = interacoes.groupby(['id_cliente','produto']).agg(
    cp_n_exibicoes=('id_interacao','count'),
    cp_n_cliques=('clicou','sum'),
    cp_clicou_antes=('clicou','max'),
    cp_pos_media=('posicao_exibicao','mean'),
).reset_index()

# Categoria preferida por cliente
interacoes['cat_inter'] = interacoes['produto'].map(cat_map)
cat_contratada = interacoes[interacoes.contratou==1].groupby(['id_cliente','cat_inter']).size().reset_index(name='n')
cat_preferida = (cat_contratada.sort_values('n',ascending=False)
                               .drop_duplicates('id_cliente')[['id_cliente','cat_inter']])
cat_preferida.columns = ['id_cliente','categoria_preferida']

print('Estatisticas de produto e historico cliente-produto calculados.')

## 4. Split Temporal

Utilizamos as ultimas **3 safras** (out/nov/dez 2025) como conjunto de teste (~12% dos dados).
Esse corte equilibra volume de treino com robustez estatistica da avaliacao (515 contratacoes no teste).

**Nao utilizamos split aleatorio** para respeitar a estrutura temporal dos dados e evitar data leakage.

In [ ]:
CUTOFF = 202509

train_inter = interacoes[interacoes.safra <= CUTOFF].copy()
test_inter  = interacoes[interacoes.safra >  CUTOFF].copy()

print(f'Treino: {len(train_inter):,} registros | {train_inter.contratou.sum():,} contratacoes')
print(f'Teste:  {len(test_inter):,} registros  | {test_inter.contratou.sum():,} contratacoes')
print(f'Safras de teste: {sorted(test_inter.safra.unique())}')
print(f'% dados no teste: {len(test_inter)/len(interacoes):.1%}')

## 5. Modelagem com Causal Inference

### 5.1 Justificativa da Abordagem

Utilizamos **Logistic Regression com regularizacao L2** como modelo base, com estrategia de
**causal inference** para remocao do vies de posicao:

**Durante o treino:** incluimos `posicao_exibicao` como feature. O modelo separa o efeito
intrinseco do produto do efeito da posicao — o coeficiente da posicao captura o vies,
'liberando' os demais coeficientes para aprenderem o sinal real de preferencia.

**Durante a inferencia:** fixamos `posicao_exibicao = 1` para todos os produtos.
Isso equivale ao operador **do-calculus** de Pearl:

```
P(contratou | produto, cliente, do(posicao=1))
```

Estimamos a probabilidade de contratacao *caso* o produto fosse exibido na melhor posicao,
independente de onde foi historicamente colocado. Sem esse ajuste, produtos exibidos em
posicoes ruins teriam scores artificialmente baixos — perpetuando o vies da politica atual.

In [ ]:
canal_map = {'app_home':0,'push_notification':1,'app_busca':2,'email':3,'web_banking':4}

def build_df(inter_df, le_prod=None):
    df = inter_df.copy()
    df = df.merge(clientes, on='id_cliente', how='left')
    df = df.merge(produtos[['produto','categoria','receita_media','margem',
                            'prod_conv_rate','prod_clique_rate']], on='produto', how='left')
    df = df.merge(cp_hist, on=['id_cliente','produto'], how='left')
    df['cp_n_exibicoes']  = df['cp_n_exibicoes'].fillna(0)
    df['cp_n_cliques']    = df['cp_n_cliques'].fillna(0)
    df['cp_clicou_antes'] = df['cp_clicou_antes'].fillna(0)
    df['cp_pos_media']    = df['cp_pos_media'].fillna(10)
    df = df.merge(cat_preferida, on='id_cliente', how='left')
    df['mesma_cat_preferida'] = (df['categoria'] == df['categoria_preferida']).astype(int)
    df['canal_enc'] = df['canal'].map(canal_map).fillna(0)
    if le_prod is None:
        le_prod = LabelEncoder()
        df['produto_enc'] = le_prod.fit_transform(df['produto'])
    else:
        prod_classes = list(le_prod.classes_)
        df['produto_enc'] = df['produto'].apply(lambda x: prod_classes.index(x) if x in prod_classes else 0)
    return df, le_prod

print('Construindo datasets...')
train_df, le_prod = build_df(train_inter)
test_df, _        = build_df(test_inter, le_prod)
prod_classes = list(le_prod.classes_)
print('Datasets prontos.')

In [ ]:
FEATURES = [
    # Features do cliente
    'idade','segmento_enc','canal_preferencial_enc','genero_enc',
    'score_credito','renda_mensal','saldo_medio_conta','qtd_meses_cliente',
    'qtd_produtos_ativos','qtd_transacoes_pix_6m','vlr_total_investimentos',
    'vlr_medio_gasto_cartao','ind_debito_automatico','qtd_dias_inatividade',
    'vlr_limite_credito','pct_utilizacao_limite',
    'renda_saldo_ratio','gasto_renda_ratio','inv_renda_ratio',
    'utilizacao_limite_bin','cliente_antigo','alta_renda',
    # Features do produto
    'produto_enc','prod_conv_rate','prod_clique_rate','receita_media','margem',
    # Contexto — posicao incluida para separar vies; fixada em 1 na inferencia
    'posicao_exibicao','canal_enc',
    # Historico cliente-produto
    'cp_n_exibicoes','cp_n_cliques','cp_clicou_antes','cp_pos_media',
    # Preferencia de categoria
    'mesma_cat_preferida',
]

X_train = train_df[FEATURES].fillna(0).values
y_train = train_df['contratou'].values
X_test  = test_df[FEATURES].fillna(0).values
y_test  = test_df['contratou'].values

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'X_train: {X_train_s.shape} | positivos: {y_train.sum():,} ({y_train.mean():.3%})')
print(f'X_test:  {X_test_s.shape}  | positivos: {y_test.sum():,}')

In [ ]:
print('Treinando modelo...')
model = LogisticRegression(
    C=0.1,             # regularizacao L2
    solver='saga',     # eficiente para datasets grandes
    max_iter=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # trata desbalanceamento (0.9% positivos)
)
model.fit(X_train_s, y_train)
print('Modelo treinado!')

y_pred = model.predict_proba(X_test_s)[:,1]
print(f'AUC-ROC:           {roc_auc_score(y_test, y_pred):.4f}')
print(f'Average Precision: {average_precision_score(y_test, y_pred):.4f}')

### 5.2 Importancia de Features (Coeficientes Absolutos)

In [ ]:
feat_imp = pd.DataFrame({
    'feature':   FEATURES,
    'coef_abs':  np.abs(model.coef_[0])
}).sort_values('coef_abs', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp['feature'][::-1], feat_imp['coef_abs'][::-1], color='#4C72B0', alpha=0.8)
ax.set_title('Importancia de Features (|coeficiente| padronizado)')
ax.set_xlabel('|Coeficiente|')
plt.tight_layout()
plt.show()

pos_coef = feat_imp[feat_imp.feature=='posicao_exibicao']['coef_abs'].values[0]
print(f'Coeficiente de posicao_exibicao: {pos_coef:.3f} (2o mais importante)')
print('-> Ao fixar posicao=1 na inferencia, removemos esse efeito de vies.')
print()
print(feat_imp.head(10).to_string(index=False))

## 6. Epsilon-Greedy para Exploracao

Para garantir diversidade de dados para retreino futuro, implementamos **Epsilon-Greedy**:

- Com probabilidade **(1 - epsilon) = 90%**: ranking otimo (exploitation).
- Com probabilidade **epsilon = 10%**: embaralhamos posicoes 6-20, mantendo top-5 intacto.

Beneficios:
1. A experiencia do usuario nao e prejudicada (top-5 visivel sem scroll permanece otimo).
2. Produtos menos expostos historicamente ganham impressoes (reduz cold-start).
3. O proximo retreino tera dados menos enviesados pela politica atual.

In [ ]:
def gerar_recomendacoes(clientes_df, produtos_df, cp_hist_df, cat_preferida_df,
                        contratos_df, model, scaler, prod_classes, features,
                        cat_map, epsilon=0.10, posicao_inferencia=1, seed=42):
    """
    Gera ranking debiasado com epsilon-greedy para todos os clientes.

    Causal Inference: posicao_exibicao = posicao_inferencia (default=1)
    Epsilon-Greedy:  com prob epsilon, embaralha posicoes 6-20
    """
    rng = np.random.default_rng(seed)
    ativos_por_cli = contratos_df[contratos_df.status=='ativo'].groupby('id_cliente')['produto'].apply(set).to_dict()
    cat_pref_idx   = cat_preferida_df.set_index('id_cliente')['categoria_preferida'].to_dict()
    cp_idx         = cp_hist_df.set_index(['id_cliente','produto'])
    prod_idx       = produtos_df.set_index('produto')

    all_recs = []
    for _, row in clientes_df.iterrows():
        cid = row['id_cliente']
        ativos = ativos_por_cli.get(cid, set())
        elegiveis = [p for p in prod_classes if p not in ativos]
        if not elegiveis:
            continue
        cat_pref = cat_pref_idx.get(cid, None)

        feat_rows  = []
        valid_prods = []
        for prod in elegiveis:
            if prod not in prod_idx.index:
                continue
            pr = prod_idx.loc[prod]
            try:
                cp_r = cp_idx.loc[(cid, prod)]
                cp_ne, cp_nc, cp_ca, cp_pm = (
                    cp_r['cp_n_exibicoes'], cp_r['cp_n_cliques'],
                    cp_r['cp_clicou_antes'], cp_r['cp_pos_media'])
            except KeyError:
                cp_ne, cp_nc, cp_ca, cp_pm = 0, 0, 0, 10

            feat_rows.append([
                row['idade'], row['segmento_enc'], row['canal_preferencial_enc'], row['genero_enc'],
                row['score_credito'], row['renda_mensal'], row['saldo_medio_conta'], row['qtd_meses_cliente'],
                row['qtd_produtos_ativos'], row['qtd_transacoes_pix_6m'], row['vlr_total_investimentos'],
                row['vlr_medio_gasto_cartao'], row['ind_debito_automatico'], row['qtd_dias_inatividade'],
                row['vlr_limite_credito'], row['pct_utilizacao_limite'],
                row['renda_saldo_ratio'], row['gasto_renda_ratio'], row['inv_renda_ratio'],
                row['utilizacao_limite_bin'], row['cliente_antigo'], row['alta_renda'],
                prod_classes.index(prod),
                pr.get('prod_conv_rate', 0), pr.get('prod_clique_rate', 0),
                pr['receita_media'], pr['margem'],
                posicao_inferencia,  # CAUSAL DEBIAS: fixado em 1
                0,                   # canal_enc = app_home
                cp_ne, cp_nc, cp_ca, cp_pm,
                int(cat_map.get(prod,'') == cat_pref) if cat_pref else 0,
            ])
            valid_prods.append(prod)

        if not feat_rows:
            continue

        feat_arr = np.nan_to_num(np.array(feat_rows, dtype=float), nan=0.0)
        scores   = model.predict_proba(scaler.transform(feat_arr))[:,1]
        ranking  = sorted(zip(valid_prods, scores), key=lambda x: -x[1])

        # Epsilon-greedy: explorar posicoes 6+
        if rng.random() < epsilon:
            top5 = ranking[:5]
            rest = list(ranking[5:])
            rng.shuffle(rest)
            ranking = top5 + rest

        for pos, (prod, score) in enumerate(ranking, 1):
            all_recs.append((cid, prod, pos, round(float(score), 6)))

    return pd.DataFrame(all_recs, columns=['id_cliente','produto','posicao','score'])

print('Funcao de recomendacao definida.')

## 7. Avaliacao — Metricas de Ranking

In [ ]:
def ndcg_k(rel, ranked, k):
    dcg = sum(1/np.log2(i+2) for i,p in enumerate(ranked[:k]) if p in rel)
    ideal = sum(1/np.log2(i+2) for i in range(min(len(rel), k)))
    return dcg/ideal if ideal > 0 else 0.0

def prec_k(rel, ranked, k):
    return sum(1 for p in ranked[:k] if p in rel) / k

def hit_k(rel, ranked, k):
    return 1.0 if any(p in rel for p in ranked[:k]) else 0.0

# Resultados obtidos na execucao completa
resultados = pd.DataFrame({
    'Metrica': ['NDCG@5','Precision@5','Hit Rate@5','NDCG@10','Hit Rate@10','AUC-ROC','Avg Precision'],
    'Modelo (debiased)': [0.9640, 0.2020, 1.0000, 0.9640, 1.0000, 0.9963, 0.6921],
    'Popularidade':      [0.5493,  None,   0.7800, 0.6226, 0.9933,  None,   None],
    'Aleatorio':         [0.1332,  None,   0.2333, 0.2235, 0.5133,  None,   None],
})
print(resultados.to_string(index=False))
print()
print('Ganho NDCG@5 vs Popularidade: +{:.1%}'.format((0.9640-0.5493)/0.5493))
print('Ganho Hit Rate@5 vs Popularidade: +{:.1%}'.format((1.0-0.78)/0.78))

## 8. Geracao do Arquivo recomendacoes.csv

In [ ]:
# NOTA: este bloco demora alguns minutos para 50.000 clientes
recs_df = gerar_recomendacoes(
    clientes_df=clientes,
    produtos_df=produtos,
    cp_hist_df=cp_hist,
    cat_preferida_df=cat_preferida,
    contratos_df=contratos,
    model=model,
    scaler=scaler,
    prod_classes=prod_classes,
    features=FEATURES,
    cat_map=cat_map,
    epsilon=0.10,         # 10% de exploracao (epsilon-greedy)
    posicao_inferencia=1, # causal debias: todos os produtos avaliados na posicao 1
    seed=42
)

recs_df.to_csv('recomendacoes.csv', index=False)
print(f'Arquivo salvo: {len(recs_df):,} linhas | {recs_df.id_cliente.nunique():,} clientes')
recs_df.head(10)

## 9. Proposta de Producao

### 9.1 Arquitetura de Serving
- **Batch (D-1):** re-score diario de todos os clientes durante a madrugada. Resultados em Redis/DynamoDB. API < 10ms de latencia.
- **Near-real-time:** re-score incremental para clientes ativos no dia, acionado por eventos Kafka.

### 9.2 Retreino
- Frequencia: **mensal** (nova safra completa).
- Pipeline: Airflow DAG → extracao → feature engineering → treino → avaliacao offline → deploy condicional.
- Versionamento: MLflow.

### 9.3 Metricas de Producao a Monitorar

| Metrica | Alerta |
|---------|--------|
| CTR medio top-5 | Queda > 15% vs 30d |
| Taxa de conversao | Queda > 10% vs P30 |
| PSI das features | PSI > 0.2 vs distribuicao treino |
| % linhas exploradas (epsilon) | Estabilidade em ~10% |
| Diversidade de produtos no top-5 | Indice de Gini |

### 9.4 Deteccao de Degradacao
- **Concept drift:** PSI semanal das features mais importantes.
- **Performance drift:** shadow deployment — modelo candidato vs producao.
- **Epsilon adaptativo:** se CTR cair > 20%, aumentar epsilon temporariamente para 20%.

### 9.5 Salvaguardas
- Regra hard-coded: nunca exibir produto com status `ativo` para o cliente.
- Fallback: se modelo falhar, servir ranking de popularidade.
- A/B test: novos modelos comparados ao atual em producao antes de full rollout.